# Análise exploratória — Casos e óbitos de COVID-19 no Brasil (2020–2022)

**Disciplina:** Linguagem de Programação - Análise e Visualização de Dados com Python (6º período, Sistemas de Informação)

**Autores:** Enzo Ribas Torres e Guilherme Ornellas Chagas

**Professor:** Alexandre Neves Louzada

**Fonte:** [Brasil.IO — Boletins epidemiológicos da COVID-19 por município por dia](https://brasil.io/dataset/covid19/caso_full/) (CC BY-SA 4.0)

Este notebook documenta a exploração preliminar, a validação do pipeline (`src/etl.py`) e a prototipação das figuras (`src/viz.py`) usadas no artigo (`docs/index.html`).

### Perguntas de pesquisa

- **RQ1.** Como evoluíram os casos e óbitos diários no Brasil entre 2020 e 2022, quantas ondas se distinguem e como variou a letalidade aparente entre elas?
- **RQ2.** Como a incidência, a mortalidade e a letalidade acumuladas se distribuíram entre UFs e regiões, e essas medidas caminham juntas?
- **RQ3.** O porte populacional do município está associado à mortalidade e à letalidade acumuladas?


In [1]:
import sys, json
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
from scipy import stats
from src import etl

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

## 1. Ingestão e inspeção do arquivo bruto

O arquivo `caso_full.csv.gz` (~92 MB comprimido, ~3,85 milhões de linhas) é lido com tipos explícitos. Como passa de 50 MB, ele **não é versionado**: `src/etl.py` o baixa automaticamente do Brasil.IO.


In [ ]:
etl.download_raw()  # não baixa de novo se o arquivo já existir
raw = etl.load_raw()
raw.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 3853648 entries, 0 to 3853647
Data columns (total 18 columns):
 #   Column                                         Dtype         
---  ------                                         -----         
 0   city                                           string        
 1   city_ibge_code                                 Int64         
 2   date                                           datetime64[us]
 3   epidemiological_week                           int32         
 4   estimated_population                           Int64         
 5   estimated_population_2019                      Int64         
 6   is_last                                        bool          
 7   is_repeated                                    bool          
 8   last_available_confirmed                       int64         
 9   last_available_confirmed_per_100k_inhabitants  float64       
 10  last_available_date                            datetime64[us]
 11  last_available_death_r

In [3]:
raw.head()

,city,city_ibge_code,date,epidemiological_week,estimated_population,estimated_population_2019,is_last,is_repeated,last_available_confirmed,last_available_confirmed_per_100k_inhabitants,last_available_date,last_available_death_rate,last_available_deaths,order_for_place,place_type,state,new_confirmed,new_deaths
0,Rio Branco,1200401,2020-03-17,202012,413418,407319,False,False,3,0.73,2020-03-17,0.00,0,1,city,AC,3,0
1,<NA>,12,2020-03-17,202012,894470,881935,False,False,3,0.34,2020-03-17,0.00,0,1,state,AC,3,0
2,Rio Branco,1200401,2020-03-18,202012,413418,407319,False,False,3,0.73,2020-03-18,0.00,0,2,city,AC,0,0
3,<NA>,12,2020-03-18,202012,894470,881935,False,False,3,0.34,2020-03-18,0.00,0,2,state,AC,0,0
4,Rio Branco,1200401,2020-03-19,202012,413418,407319,False,False,4,0.97,2020-03-19,0.00,0,3,city,AC,1,0


### 1.1 Relatório de qualidade


In [4]:
rel = etl.validate_raw(raw)
print(json.dumps(rel, ensure_ascii=False, indent=2))

{
  "fonte": "https://data.brasil.io/dataset/covid19/caso_full.csv.gz",
  "linhas_brutas": 3853648,
  "colunas": 18,
  "periodo": [
    "2020-02-25",
    "2022-03-27"
  ],
  "linhas_por_tipo": {
    "city": 3833529,
    "state": 20119
  },
  "nulos_por_coluna": {
    "city": 20119,
    "city_ibge_code": 13646,
    "estimated_population": 13646,
    "estimated_population_2019": 13646,
    "last_available_confirmed_per_100k_inhabitants": 29166
  },
  "linhas_repetidas_is_repeated": 1015645,
  "linhas_importados_indefinidos": 13646,
  "municipios_distintos": 5570,
  "ufs_distintas": 27,
  "new_confirmed_negativos": 38638,
  "new_deaths_negativos": 6261,
  "chaves_duplicadas_local_data": 0
}


**Achados da inspeção:**

- Há dois níveis de granularidade no mesmo arquivo (`place_type = city` e `state`). As análises por UF e Brasil usam somente as linhas `state`, para não contar o mesmo caso duas vezes.
- A categoria `Importados/Indefinidos` (casos sem município definido) não tem código IBGE nem população. Ela entra nos totais estaduais, mas fica fora da análise municipal.
- `is_repeated = True` (~26% das linhas) indica dias sem boletim novo, com o acumulado repetido. Por isso os novos casos se concentram no dia seguinte, e usamos **médias móveis de 7 dias**.
- Valores negativos em `new_confirmed`/`new_deaths` são **correções retroativas** das secretarias. Eles são mantidos (o acumulado depende deles) e sinalizados na coluna `e_correcao`.


In [ ]:
print("Maiores correções negativas (casos):")
raw.loc[
    raw.new_confirmed < 0,
    ["state", "city", "date", "new_confirmed", "last_available_confirmed"],
].nsmallest(5, "new_confirmed")

Maiores correções negativas (casos):


,state,city,date,new_confirmed,last_available_confirmed
2645180,RN,Nísia Floresta,2021-11-23,-59429,2542
2645009,RN,Natal,2021-11-22,-42582,60596
2873608,RS,Santa Margarida do Sul,2021-04-05,-23275,51
2873110,RS,Santa Maria,2021-04-04,-22826,500
2645188,RN,Passa e Fica,2021-11-23,-19207,1164


### 1.2 Consistência: soma dos municípios × total estadual (último dia)


In [ ]:
ult = raw[raw.date == raw.date.max()]
soma_mun = (
    ult[ult.place_type == "city"]
    .groupby("state", observed=True)
    .last_available_deaths.sum()
)
tot_uf = ult[ult.place_type == "state"].set_index("state").last_available_deaths
cmp = pd.DataFrame({"soma_municipios": soma_mun, "total_uf": tot_uf})
cmp["diferenca"] = cmp.total_uf - cmp.soma_municipios
cmp.sort_values("diferenca", ascending=False).head(8)

,soma_municipios,total_uf,diferenca
state,,,
SP,154369,167110,12741
MG,54162,60767,6605
RJ,66422,72695,6273
CE,24139,26708,2569
PR,40357,42896,2539
BA,27320,29658,2338
ES,12575,14323,1748
GO,24504,26212,1708


Os totais estaduais **superam** a soma dos municípios (incluída a categoria Importados/Indefinidos). Em SP a diferença chega a 12,7 mil óbitos (7,6%). A causa aparece abaixo: as séries **municipais** de várias UFs pararam de ser atualizadas entre ago/2021 e mar/2022, mas o total estadual continuou. Por isso:

1. as análises por UF e Brasil usam as linhas `state`;
2. as comparações entre municípios usam uma **data de corte comum** (a mais recente em que todos ainda estavam atualizados), para não misturar períodos diferentes.


In [ ]:
cid = raw[(raw.place_type == "city") & raw.is_last & (raw.city != etl.IMPORTADOS)]
print("Data de corte comum:", cid.last_available_date.min().date())
cid.groupby("state", observed=True).last_available_date.agg(
    ["min", "median", "max"]
).sort_values("median")

Data de corte comum: 2021-08-22


,min,median,max
state,,,
MS,2021-08-22,2021-08-22,2021-08-22
PB,2021-08-22,2021-08-22,2021-08-22
CE,2021-09-16,2021-09-16,2021-09-16
MA,2021-09-16,2021-09-16,2021-09-16
MG,2021-09-22,2021-09-22,2021-09-22
RO,2021-09-30,2021-09-30,2021-09-30
RJ,2021-10-02,2021-10-02,2021-10-02
ES,2021-10-03,2021-10-03,2021-10-03
PE,2021-10-22,2021-10-22,2021-10-22


## 2. Execução do pipeline e tabelas processadas


In [8]:
tabelas = etl.run()
{k: v.shape for k, v in tabelas.items()}

{'brasil_diario': (762, 10),
 'uf_diario': (20119, 14),
 'uf_mensal': (676, 9),
 'uf_resumo': (27, 9),
 'municipios': (5570, 16)}

In [ ]:
br, uf_res, uf_mes, mun = (
    tabelas[k] for k in ("brasil_diario", "uf_resumo", "uf_mensal", "municipios")
)
br.tail()

,data,casos_novos,obitos_novos,casos_acumulados,obitos_acumulados,ufs_reportando,populacao,casos_mm7,obitos_mm7,letalidade_defasada_14d
757,2022-03-23,46255,283,29739457,658293,27,211755692,"35,122.70",292.30,0.58
758,2022-03-24,37290,308,29776747,658601,27,211755692,"33,943.30",268.10,0.54
759,2022-03-25,34009,257,29810756,658858,27,211755692,"31,947.90",250.30,0.53
760,2022-03-26,28673,192,29839429,659050,27,211755692,"30,588.00",237.00,0.52
761,2022-03-27,10311,109,29849740,659159,27,211755692,"30,194.70",237.40,0.52


## 3. RQ1 — Evolução temporal e ondas


In [ ]:
ondas = {
    "1ª onda": ("2020-03-01", "2020-10-31"),
    "2ª onda (Gama)": ("2020-11-01", "2021-10-31"),
    "3ª onda (Ômicron)": ("2021-11-01", "2022-03-27"),
}
linhas = []
for nome, (a, b) in ondas.items():
    s = br[(br.data >= a) & (br.data <= b)]
    ic, io = s.casos_mm7.idxmax(), s.obitos_mm7.idxmax()
    linhas.append(
        {
            "onda": nome,
            "pico_casos_mm7": s.casos_mm7.max(),
            "data_pico_casos": br.data[ic].date(),
            "pico_obitos_mm7": s.obitos_mm7.max(),
            "data_pico_obitos": br.data[io].date(),
            "casos": s.casos_novos.sum(),
            "obitos": s.obitos_novos.sum(),
            "letalidade_%": s.obitos_novos.sum() / s.casos_novos.sum() * 100,
        }
    )
pd.DataFrame(linhas).set_index("onda")

,pico_casos_mm7,data_pico_casos,pico_obitos_mm7,data_pico_obitos,casos,obitos,letalidade_%
onda,,,,,,,
1ª onda,"47,514.60",2020-07-28,"1,095.10",2020-07-25,5537761,159972,2.89
2ª onda (Gama),"78,663.30",2021-06-22,"3,112.30",2021-04-11,16276862,448109,2.75
3ª onda (Ômicron),"189,243.40",2022-01-29,894.30,2022-02-12,8035115,51078,0.64


In [ ]:
mes = (
    br.assign(mes=br.data.dt.to_period("M"))
    .groupby("mes")[["casos_novos", "obitos_novos"]]
    .sum()
)
mes["letalidade_%"] = mes.obitos_novos / mes.casos_novos * 100
mes.loc["2020-03":]

,casos_novos,obitos_novos,letalidade_%
mes,,,
2020-03,5822,202,3.47
2020-04,81302,5778,7.11
2020-05,428968,23387,5.45
2020-06,896570,30424,3.39
2020-07,1257787,32936,2.62
2020-08,1244378,28891,2.32
2020-09,902536,22346,2.48
2020-10,720398,16008,2.22
2020-11,804202,13296,1.65


**Leitura:** há três ondas bem distintas. A segunda (Gama) concentrou **68% dos óbitos** do período, com pico de 3.112 óbitos/dia (MM7) em 11/04/2021. A terceira (Ômicron) bateu o recorde de casos (189 mil/dia em 29/01/2022), mas teve letalidade aparente de 0,64%, contra 2,75% na segunda. Isso é compatível com a cobertura vacinal já alta e com a menor gravidade intrínseca da variante.


## 4. RQ2 — Desigualdades entre UFs e regiões


In [12]:
uf_res

,uf,regiao,populacao,casos_acumulados,obitos_acumulados,data_referencia,incidencia_por_100k,mortalidade_por_100k,letalidade_pct
0,MT,Centro-Oeste,3526220,724653,14854,2022-03-27,"20,550.42",421.24,2.05
1,RJ,Sudeste,17366189,2078817,72695,2022-03-27,"11,970.48",418.60,3.50
2,RO,Norte,1796460,391943,7172,2022-03-27,"21,817.52",399.23,1.83
3,DF,Centro-Oeste,3055149,691353,11573,2022-03-27,"22,629.11",378.80,1.67
4,MS,Centro-Oeste,2809394,523565,10486,2022-03-27,"18,636.23",373.25,2.00
5,PR,Sul,11516840,2407960,42896,2022-03-27,"20,908.17",372.46,1.78
6,GO,Centro-Oeste,7113540,1269853,26212,2022-03-27,"17,851.21",368.48,2.06
7,SP,Sudeste,46289333,5232374,167110,2022-03-27,"11,303.63",361.01,3.19
8,ES,Sudeste,4064052,1037188,14323,2022-03-27,"25,521.03",352.43,1.38
9,RS,Sul,11422973,2263880,38985,2022-03-27,"19,818.66",341.29,1.72


In [ ]:
reg = uf_res.groupby("regiao", observed=True).agg(
    pop=("populacao", "sum"),
    casos=("casos_acumulados", "sum"),
    obitos=("obitos_acumulados", "sum"),
)
reg["incidencia_100k"] = reg.casos / reg["pop"] * 1e5
reg["mortalidade_100k"] = reg.obitos / reg["pop"] * 1e5
reg["letalidade_%"] = reg.obitos / reg.casos * 100
reg.sort_values("mortalidade_100k", ascending=False)

,pop,casos,obitos,incidencia_100k,mortalidade_100k,letalidade_%
regiao,,,,,,
Centro-Oeste,16504303,3209424,63125,"19,445.98",382.48,1.97
Sudeste,89012240,11665780,314895,"13,105.82",353.77,2.70
Sul,30192315,6343015,103529,"21,008.71",342.90,1.63
Norte,18672591,2466006,49802,"13,206.55",266.71,2.02
Nordeste,57374243,6165515,127808,"10,746.14",222.76,2.07


In [ ]:
for x, y in [
    ("incidencia_por_100k", "mortalidade_por_100k"),
    ("incidencia_por_100k", "letalidade_pct"),
    ("mortalidade_por_100k", "letalidade_pct"),
]:
    r = stats.spearmanr(uf_res[x], uf_res[y])
    print(f"Spearman {x} x {y}: rho = {r.statistic:+.3f}  (p = {r.pvalue:.2g})")

Spearman incidencia_por_100k x mortalidade_por_100k: rho = +0.661  (p = 0.00017)
Spearman incidencia_por_100k x letalidade_pct: rho = -0.736  (p = 1.2e-05)
Spearman mortalidade_por_100k x letalidade_pct: rho = -0.040  (p = 0.84)


**Leitura:** a mortalidade variou 2,8 vezes entre a UF mais afetada (MT, 421 por 100 mil) e a menos afetada (MA, 153 por 100 mil). O dado mais revelador é a **correlação negativa forte entre incidência e letalidade** (ρ ≈ −0,74). UFs que notificaram mais casos por habitante tiveram letalidade _aparente_ menor. Isso sugere que a letalidade medida reflete a capacidade de testagem (subnotificação de casos leves), não só a gravidade da doença. RJ e SP, com letalidade acima de 3%, são o exemplo mais claro.


## 5. RQ3 — Porte populacional dos municípios


In [ ]:
mun[
    [
        "populacao",
        "casos_acumulados",
        "obitos_acumulados",
        "incidencia_por_100k",
        "mortalidade_por_100k",
        "letalidade_pct",
    ]
].describe().T

,count,mean,std,min,25%,50%,75%,max
populacao,"5,570.00","38,017.18","222,892.99",776.00,"5,442.25","11,665.50","25,663.75","12,325,232.00"
casos_acumulados,"5,570.00","3,618.00","18,533.60",10.00,428.25,932.50,"2,297.50","929,136.00"
obitos_acumulados,"5,570.00",101.29,753.60,0.00,8.00,20.00,50.00,"36,714.00"
incidencia_por_100k,"5,570.00","9,453.32","4,603.07",178.95,"6,075.77","9,009.74","12,271.19","46,807.43"
mortalidade_por_100k,"5,570.00",203.23,112.79,0.00,119.38,185.93,271.43,784.52
letalidade_pct,"5,570.00",2.41,1.57,0.00,1.49,2.12,2.96,32.10


In [ ]:
g = mun.groupby("porte", observed=True).agg(
    municipios=("codigo_ibge", "size"),
    populacao=("populacao", "sum"),
    obitos=("obitos_acumulados", "sum"),
    casos=("casos_acumulados", "sum"),
    mortalidade_mediana=("mortalidade_por_100k", "median"),
    mortalidade_iqr=(
        "mortalidade_por_100k",
        lambda s: s.quantile(0.75) - s.quantile(0.25),
    ),
    letalidade_mediana=("letalidade_pct", "median"),
)
g["mortalidade_agregada"] = g.obitos / g.populacao * 1e5
g["letalidade_agregada_%"] = g.obitos / g.casos * 100
g

,municipios,populacao,obitos,casos,mortalidade_mediana,mortalidade_iqr,letalidade_mediana,mortalidade_agregada,letalidade_agregada_%
porte,,,,,,,,,
Até 20 mil,3783,31797750,60018,2893941,173.81,148.82,2.02,188.75,2.07
20–50 mil,1110,33828245,69549,3162700,186.84,140.55,2.20,205.59,2.20
50–100 mil,351,24150422,57139,2445663,226.06,134.01,2.35,236.60,2.34
100–500 mil,277,54457497,155878,5537430,276.84,124.01,2.89,286.24,2.81
Mais de 500 mil,49,67521778,221588,6112517,316.79,101.48,3.31,328.17,3.63


In [ ]:
kw = stats.kruskal(
    *[d.mortalidade_por_100k for _, d in mun.groupby("porte", observed=True)]
)
print(
    f"Kruskal-Wallis (mortalidade ~ porte): H = {kw.statistic:.1f}, p = {kw.pvalue:.2g}"
)
for y in ["mortalidade_por_100k", "letalidade_pct", "incidencia_por_100k"]:
    r = stats.spearmanr(mun.populacao, mun[y], nan_policy="omit")
    print(f"Spearman populacao x {y}: rho = {r.statistic:+.3f} (p = {r.pvalue:.2g})")
cap = mun.groupby("e_capital").agg(
    pop=("populacao", "sum"),
    obitos=("obitos_acumulados", "sum"),
    casos=("casos_acumulados", "sum"),
)
cap["mortalidade_100k"] = cap.obitos / cap["pop"] * 1e5
cap["letalidade_%"] = cap.obitos / cap.casos * 100
cap

Kruskal-Wallis (mortalidade ~ porte): H = 297.2, p = 4.5e-63
Spearman populacao x mortalidade_por_100k: rho = +0.131 (p = 8.6e-23)
Spearman populacao x letalidade_pct: rho = +0.156 (p = 8e-32)
Spearman populacao x incidencia_por_100k: rho = -0.012 (p = 0.37)


,pop,obitos,casos,mortalidade_100k,letalidade_%
e_capital,,,,,
False,161221137,393423,15364373,244.03,2.56
True,50534555,170749,4787878,337.89,3.57


### 5.1 Outliers (regra de Tukey, 1,5 × IQR)


In [ ]:
q1, q3 = mun.mortalidade_por_100k.quantile([0.25, 0.75])
lim = q3 + 1.5 * (q3 - q1)
out = mun[mun.mortalidade_por_100k > lim]
print(f"Limite superior: {lim:.1f} por 100 mil — {len(out)} municípios acima")
out.porte.value_counts(sort=False)

Limite superior: 499.5 por 100 mil — 68 municípios acima


porte
Até 20 mil         57
20–50 mil           7
50–100 mil          1
100–500 mil         2
Mais de 500 mil     1
Name: count, dtype: int64

In [ ]:
out.nlargest(10, "mortalidade_por_100k")[
    ["municipio", "uf", "populacao", "obitos_acumulados", "mortalidade_por_100k"]
]

,municipio,uf,populacao,obitos_acumulados,mortalidade_por_100k
3599,Meridiano,SP,3824,30,784.52
3674,Parisi,SP,2169,17,783.77
5378,Campos Verdes,GO,1830,14,765.03
4228,Salto do Itararé,PR,4898,37,755.41
4468,Marema,SC,1750,13,742.86
3321,Bálsamo,SP,9139,66,722.18
3916,Altamira do Paraná,PR,1682,12,713.44
3782,Santa Clara d'Oeste,SP,2113,15,709.89
4924,Pirapó,RS,2254,16,709.85
42,Pimenteiras do Oeste,RO,2148,15,698.32


**Leitura (corte comum em 22/08/2021):** a mortalidade mediana sobe de forma monotônica com o porte, de 174 por 100 mil (até 20 mil hab.) para 317 por 100 mil (mais de 500 mil hab.). A diferença é significativa (Kruskal-Wallis, H ≈ 297, p < 10⁻⁶⁰). A **incidência**, por outro lado, não tem relação com o porte (ρ ≈ −0,01, p = 0,37). O excesso de mortalidade nos municípios grandes, portanto, vem da **letalidade aparente** maior (2,07% contra 3,63% nas taxas agregadas), e não de mais casos por habitante. A dispersão cai com o porte: 57 dos 68 outliers estão em municípios de até 20 mil habitantes, onde poucos óbitos já mexem muito na taxa. As capitais têm mortalidade 38% maior que os demais municípios (338 contra 244 por 100 mil). A associação contínua é fraca (ρ ≈ 0,13), ou seja, o porte é **um** fator entre vários.


## 6. Geração das figuras do artigo


In [ ]:
from src import viz

viz.main()
sorted(p.name for p in viz.ASSETS.iterdir())

13:58:14 INFO Gerando figuras em docs/assets


13:58:14 INFO   fig1_evolucao_brasil.html


13:58:15 INFO   fig1_evolucao_brasil.png


13:58:15 INFO   fig2_letalidade_mensal.html


13:58:15 INFO   fig2_letalidade_mensal.png


13:58:15 INFO   fig3_heatmap_uf_mes.html


... (4 outputs left out)

13:58:16 INFO   fig5_incidencia_letalidade.png


13:58:16 INFO   fig6_porte_municipios.html


13:58:16 INFO   fig6_porte_municipios.png


['fig1_evolucao_brasil.html',
 'fig1_evolucao_brasil.png',
 'fig2_letalidade_mensal.html',
 'fig2_letalidade_mensal.png',
 'fig3_heatmap_uf_mes.html',
 'fig3_heatmap_uf_mes.png',
 'fig4_ranking_uf.html',
 'fig4_ranking_uf.png',
 'fig5_incidencia_letalidade.html',
 'fig5_incidencia_letalidade.png',
 'fig6_porte_municipios.html',
 'fig6_porte_municipios.png']